## WARNING: This notebook is designed to run on the PSA graph and its variations. Aside from the base case, no other graphs will work in combination with this notebook.

In [1]:
import sys
%pip install -r "./venv/requirements.txt"
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
import numpy as np
import datetime
from causalbo.do_calculus import E_output_given_do
import concurrent.futures
import pickle


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Auxiliary code for plotting

In [2]:
def generate_trend_plot(CBO_costs, CBO_opts, MISO_costs, MISO_opts, CBO_s_costs, CBO_s_opts, graph = None, opt = None, maximize = False):
    # Assuming n_runs, run_costs_std, run_opts_std, run_costs_cbo, run_opts_cbo are already defined
    assert len(CBO_costs) == len(CBO_opts) == len(MISO_costs) == len(MISO_opts) == len(CBO_s_costs) == len(CBO_s_opts), "Result inputs must contain an equal amount of records"

    n_runs = len(MISO_opts)

    # Define x and y values for multiple lines
    lines = [None] * n_runs
    lines_cbo = [None] * n_runs
    lines_cbo_s = [None] * n_runs

    for i in range(n_runs):
        lines[i] = [MISO_costs[i], MISO_opts[i]]
        lines_cbo[i] = [CBO_costs[i], CBO_opts[i]]
        lines_cbo_s[i] = [CBO_s_costs[i], CBO_s_opts[i]]

    colors = ["#FF7F0E", "#FFBB78", "#FFDAB9"]  # Orange shades
    colors_cbo = ["#1F77B4", "#A6C8E0", "#B0E0E6"]  # Light blue shades
    colors_std = ["#2CA02C", "#98DF8A", "#A9DFBF"]  # Green shades


    # Create a single plot
    fig_object = plt.figure(figsize=(10, 6))

    # Function to plot lines and find min/max/average
    def plot_lines(linelist, label_prefix, colors):

        # Collect the individual coordinates
        coords = []

        for run in linelist:
            for i in range(len(run[0])):
                coords.append([run[0][i], run[1][i]])
    
        y_max = []
        y_min = []
    
        if maximize:

            # Using sort() to sort ascending in-place, key on the y coordinate
            coords.sort(key=lambda x: (x[1],-x[0]))

            cur_max = float('-inf')
            cur_max_cost = -1

            # Find the worst performing run
            for coord in coords:
                if coord[1] >= cur_max and coord[0] > cur_max_cost:
                    y_min.append(coord)
                    cur_max = coord[1]
                    cur_max_cost = coord[0]

            coords.sort(key=lambda x: (-x[1],x[0]))
            cur_max = float('inf')
            cur_max_cost = float('inf')
            # Find the best performing run
            for coord in coords:
                if coord[1] <= cur_max and coord[0] < cur_max_cost:
                    y_max.append(coord)
                    cur_max = coord[1]
                    cur_max_cost = coord[0]

            y_min.insert(0,[0,y_min[0][1]])
            y_max.insert(len(y_max),[0,y_min[0][1]])
            y_max.reverse()

        else:
            cur_max = float('inf')
            cur_max_cost = -1

            # Using sort() to sort descending in-place
            coords.sort(key=lambda x: (-x[1],-x[0]))

            # Find the worst performing run
            for coord in coords:
                if coord[1] <= cur_max and coord[0] > cur_max_cost:
                    y_max.append(coord)
                    cur_max = coord[1]
                    cur_max_cost = coord[0]

            # Using sort() to sort descending in-place
            coords.sort(key=lambda x: (x[1],x[0]))
            cur_max = float('-inf')
            cur_max_cost = float('inf')

            # Find the best performing run
            for coord in coords:
                if coord[1] >= cur_max and coord[0] < cur_max_cost:
                    y_min.append(coord)
                    cur_max = coord[1]
                    cur_max_cost = coord[0]

            y_max.insert(0,[0,y_max[0][1]])
            y_min.insert(len(y_min),[0,y_min[len(y_min)-1][1]])
            y_min.reverse()

                    
        min_data = list(zip(*y_min))
        max_data = list(zip(*y_max))

        x1, x2, y1, y2 = list(min_data[0]), list(max_data[0]), list(min_data[1]), list(max_data[1])
        # Create a common x-axis grid
        x_combined = np.sort(np.concatenate([x1, x2])) 
        
        # Interpolate each line to the common x-values
        interpolated_lines = []
        for run in linelist:
            x, y = run
            interp_y = np.interp(x_combined, x, y)
            interpolated_lines.append(interp_y)

        y_avg = np.mean(interpolated_lines, axis=0)

        # Interpolate y-values for both lines on the common x-axis
        y1_interp = np.interp(x_combined, x1, y1)
        y2_interp = np.interp(x_combined, x2, y2)

        # # Fill the area between the min and max lines
        plt.fill_between(x_combined, y1_interp, y2_interp, color=colors[2], alpha=0.5)

        # Plot the min and max lines
        plt.plot([coord[0] for coord in y_min], [coord[1] for coord in y_min], color=colors[0], linewidth=2)
        plt.plot([coord[0] for coord in y_max], [coord[1] for coord in y_max], label=f'{label_prefix}', color=colors[0], linewidth=2)
        plt.plot(x_combined, y_avg, color=colors[1], linestyle=':', linewidth=2)  # Plot the average line

    def plot_individuals(linelist, label_prefix, colors):

        # Collect the individual coordinates
        interpolated_lines = []
        x_combined = []
        for run in linelist:
            x_combined.extend(run[0])

        x_combined = np.sort(x_combined)

        for run in linelist:
            for i in range(len(run[0])):
                x, y = run
                interp_y = np.interp(x_combined, x, y)
                interpolated_lines.append(interp_y)
                plt.plot(x, y, color=colors[0], linewidth=0.3, alpha=0.05)

        y_avg = np.mean(interpolated_lines, axis=0)

        plt.plot(x_combined, y_avg, label=f'{label_prefix}', color=colors[0],linewidth=3)

    # Plot both sets of lines
    plot_lines(lines, 'MSBO', colors_std)
    plot_lines(lines_cbo, 'MSCBO', colors)
    plot_lines(lines_cbo_s, 'CBO', colors_cbo)

    if graph.__class__.__name__ == 'PSAGraph':
            plt.axhline(y = opt, color = 'r', linestyle = '-', label='True optimum',linewidth=3) 
    
    # elif graph.__class__.__name__ == 'ToyGraph':
    #         global_optimum = torch.tensor([E_output_given_do(interventional_variable=['Z'], interventional_value=torch.Tensor([-3.2]), causal_model=graph.true_graph)]).item()
    #         plt.axhline(y = global_optimum, color = 'r', linestyle = '-', label='True optimum') 
    

    # Adding labels and legend
    plt.title('MSCBO, CBO and MSBO trend lines')
    # plt.xlabel('Total Cost\nObservation costs 1 unit per point, intervention costs 10 units per variable')
    plt.xlabel('Total Cost')
    plt.ylabel('global optimum')
    plt.legend()
    plt.grid(True)

    now = datetime.datetime.now()

    pickle.dump(fig_object, open(f"./causalbo/result_plots/{graph.__class__.__name__}_{now.strftime('%Y%m%d_%H%M%S')}_domains.pickle","wb"))
    plt.savefig(f"./causalbo/result_plots/{graph.__class__.__name__}_{now.strftime('%Y%m%d_%H%M%S')}_domains.pdf", format="pdf")

    # Show the plot
    plt.show()

    fig_object_2 = plt.figure(figsize=(10, 6))

    # Plot the individual line plots
    plot_individuals(lines, 'MSBO', colors_std)
    plot_individuals(lines_cbo, 'MSCBO', colors)
    plot_individuals(lines_cbo_s, 'CBO', colors_cbo)

    if graph.__class__.__name__ == 'PSAGraph':
        # global_optimum = torch.tensor([E_output_given_do(interventional_variable=['ASPIRIN','STATIN'], interventional_value=torch.Tensor([0.0,1.0]), causal_model=graph.true_graph)]).item()
        # plt.axhline(y = global_optimum, color = 'r', linestyle = '-', label='True optimum') 
        plt.axhline(y=opt, color='r', label='True optimum', linestyle='-',linewidth=3)
    
    # elif graph.__class__.__name__ == 'ToyGraph':
    #         global_optimum = torch.tensor([E_output_given_do(interventional_variable=['Z'], interventional_value=torch.Tensor([-3.2]), causal_model=graph.true_graph)]).item()
    #         plt.axhline(y = global_optimum, color = 'r', linestyle = '-', label='True optimum') 
    
    # Adding labels and legend
    plt.title('MSCBO, CBO and MCBO trend lines')
    # plt.xlabel('Total Cost\nObservation costs 1 unit per point, intervention costs 10 units per variable')
    plt.xlabel('Total Cost')
    plt.ylabel('global optimum')
    plt.legend()
    plt.grid(True)
       
    # Show the plot
    pickle.dump(fig_object_2, open(f"./causalbo/result_plots/{graph.__class__.__name__}_{now.strftime('%Y%m%d_%H%M%S')}_lines.pickle","wb"))
    plt.savefig(f"./causalbo/result_plots/{graph.__class__.__name__}_{now.strftime('%Y%m%d_%H%M%S')}_lines.pdf", format="pdf")
    plt.show()


def generate_individual_plots(CBO_costs, CBO_opts, MISO_costs, MISO_opts):

    assert len(CBO_costs) == len(CBO_opts) == len(MISO_costs) == len(MISO_opts), "Result inputs must contain an equal amount of records"

    n_runs = len(MISO_opts)

    for i in range(n_runs):
        plt.plot([0] + MISO_costs[i], [MISO_opts[i][0]] + MISO_opts[i], "-o")
        plt.plot(CBO_costs[i], CBO_opts[i], '-o')
        plt.legend(['Standard GP', 'Causal GP'])
        plt.xlabel("Total Cost\nObservation costs 1 unit per point, intervention costs 10 units per variable")
        plt.ylabel("Global Optimum")
        plt.title("Standard GP vs Causal GP: Toy Graph, 10 iterations")
        plt.show()

    


# Threaded Optimizer for single-source CBO


In [3]:
from causalbo.cbo_loop_threading import CBOLoop

def optim_threaded(args):
        source, true_graph, n_obs, num_iterations, intervention_cost, type_trial, exploration_set, f_init = args

        (global_optimum, global_optimal_set, gp, D_i, D_o, cost_over_time_causal, global_optimum_over_time_causal) = CBOLoop(
                                observational_samples=source.observational_samples,
                                graph=source.graph,
                                exploration_set=exploration_set,
                                num_steps=num_iterations,
                                num_initial_obs=n_obs,
                                num_obs_per_step=20,
                                num_max_allowed_obs=1000,
                                intervention_cost=intervention_cost,
                                interventional_domain=source.interventional_domain,
                                type_trial=type_trial,
                                pomis_criterion='distance',
                                objective_function=true_graph,
                                early_stopping_iters=10, verbose=False, preset_opt=f_init,
                                cutoff_criterion="cost", cost_cutoff=1200)

        idx = global_optimum_over_time_causal.index(global_optimum) + 1
        global_optimum_over_time_causal = [global_optimum_over_time_causal[0]] + global_optimum_over_time_causal[:idx]
        cost_over_time_causal = [0] + cost_over_time_causal[:idx]

        return(cost_over_time_causal,global_optimum_over_time_causal)

# Multi-source CBO Loop


## Base case: Same DAG

In [4]:
# with open("./causalbo/result_plots/PSAGraph_20250716_020428_ground_truth.pickle", "rb") as f:
#     toy_graph_true = pickle.load(f)

# opt = torch.min(toy_graph_true.sample(n=1000000,do={"ASPIRIN": 0.0,"STATIN": 1.0})["PSA"]).item()
# f_init = max(toy_graph_true.observational_samples[toy_graph_true.graph.output_node].values)

# opt = 3.95

# n_runs = 10
# n = 0
# run_ss_cbo = True # Possibility to run single-source CBO as well as a means of comparison

# run_opts_cbo = [None] * n_runs
# run_costs_cbo = [None] * n_runs
# run_opts_std = [None] * n_runs
# run_costs_std = [None] * n_runs
# run_opts_cbo_single = [None] * n_runs
# run_costs_cbo_single = [None] * n_runs

# source_fidelities = [1,1]
# num_iterations = 10  # Number of iterations or total evaluations
# max_fails = 5 # Maximum number of failed runs
# fails = 0
# minimize = True

In [ ]:
from causalbo.cbo_loop_multisource import CBOLoopMultiSource
from causalbo.mcbo import multi_source_optimization
from causalbo.do_calculus import E_output_given_do
from matplotlib import pyplot as plt
from causalbo.sample_data.bi_graph import BiGraph
from causalbo.sample_data.mis_graph import MIStesterGraph
from causalbo.sample_data.psa_graph import PSAGraph
from causalbo.sample_data.toy_graph import ToyGraph
from causalbo.sample_data.bee_graph_base import HiveGraph
from copy import deepcopy
import time
import warnings
warnings.filterwarnings('ignore')

NUM_INITIAL_OBSERVATIONS = 30 
INTERVENTION_COST = 20
OBSERVATION_COST = 1

# PSAGraph object is a prebuilt class containing the medical dataset.
# toy_graph_true = PSAGraph()
# now = datetime.datetime.now()
# pickle.dump(toy_graph_true, open(f"./causalbo/result_plots/{toy_graph_true.__class__.__name__}_{now.strftime('%Y%m%d_%H%M%S')}_ground_truth.pickle","wb"))

with open("./causalbo/result_plots/PSAGraph_20250716_020428_ground_truth.pickle", "rb") as f:
    toy_graph_true = pickle.load(f)

# opt = torch.min(toy_graph_true.sample(n=1000000,do={"ASPIRIN": 0.0,"STATIN": 1.0})["PSA"]).item()
f_init = max(toy_graph_true.observational_samples[toy_graph_true.graph.output_node].values)

opt = 3.95

# pickle.dump(opt, open(f"./causalbo/result_plots/PSA_opt.pickle","wb"))



toy_graph = PSAGraph()
# toy_graph.observational_samples = toy_graph.objective_samples
# toy_graph_2 = deepcopy(toy_graph)
toy_graph_2 = PSAGraph()
sources = [toy_graph,toy_graph_2]

n_runs = 10
n = 0
run_ss_cbo = True # Possibility to run single-source CBO as well as a means of comparison

run_opts_cbo = [None] * n_runs
run_costs_cbo = [None] * n_runs
run_opts_std = [None] * n_runs
run_costs_std = [None] * n_runs
run_opts_cbo_single = [None] * n_runs
run_costs_cbo_single = [None] * n_runs

sources = [toy_graph, toy_graph_2]  # Assume 2 different information sources
source_fidelities = [1,1]
num_iterations = 10  # Number of iterations or total evaluations
max_fails = 5 # Maximum number of failed runs
fails = 0
minimize = True

# We wrap the method in a try except block to resolve stability issues and make sure we get enough runs in
# We wrap the method in a try except block to resolve stability issues and make sure we get enough runs in
while n < n_runs:
        try:
                # Store total cost
                total_cost_standard = NUM_INITIAL_OBSERVATIONS
                # Store changes in cost over time
                cost_over_time_standard = []
                # Store optimum over time
                global_optimum_over_time_standard = []

                t0 = time.time()
                # Initialize and run Causal GP using the CBO algorithm from Aglietti et. al.
                (global_optimum, global_optimal_set, global_optimal_source, gp, D_i, D_o, cost_over_time_causal, global_optimum_over_time_causal) = CBOLoopMultiSource(
                        observational_samples=[toy_graph.observational_samples, toy_graph_2.observational_samples],
                        graphs=[toy_graph.graph,toy_graph_2.graph],
                        source_costs = source_fidelities,
                        exploration_sets=[None, None],
                        num_steps=num_iterations,
                        num_initial_obs=[NUM_INITIAL_OBSERVATIONS, NUM_INITIAL_OBSERVATIONS],
                        observation_cost=OBSERVATION_COST,
                        intervention_cost=INTERVENTION_COST,
                        num_obs_per_step=20,
                        num_max_allowed_obs=1000,
                        interventional_domains=[toy_graph.interventional_domain,toy_graph_2.interventional_domain],
                        type_trial='min',
                        pomis_criterion='distance',
                        objective_function=toy_graph_true,
                        early_stopping_iters=10, 
                        verbose=False,
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200)

                idx = global_optimum_over_time_causal.index(global_optimum) + 1
                global_optimum_over_time_causal = [global_optimum_over_time_causal[0]] + global_optimum_over_time_causal[:idx]
                cost_over_time_causal = [0] + cost_over_time_causal[:idx]

                run_opts_cbo[n] = global_optimum_over_time_causal
                run_costs_cbo[n] = cost_over_time_causal

                t1 = time.time()
                print(f'\nTotal time elapsed in seconds: {t1-t0}')
                print(f'\nOptimal source after BO = {global_optimal_source}')

                # Multi-source misokg
                (multi_source_gp, global_optimum_over_time_standard, cost_over_time_standard) = multi_source_optimization(
                        sources=sources, 
                        source_costs=source_fidelities, 
                        ground_truth=toy_graph_true,
                        budget=num_iterations, 
                        num_obs=NUM_INITIAL_OBSERVATIONS,
                        intervention_cost=INTERVENTION_COST,
                        total_cost_standard=[NUM_INITIAL_OBSERVATIONS,NUM_INITIAL_OBSERVATIONS],
                        type_trial='min',
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200
                        )


                run_opts_std[n] = global_optimum_over_time_standard
                run_costs_std[n] = cost_over_time_standard

                # Single source CBO
                if run_ss_cbo:
                        cost_results = [None] * len(sources)
                        opt_results = [None] * len(sources)
                        with concurrent.futures.ThreadPoolExecutor() as executor:
                                futures = [executor.submit(optim_threaded, (source, toy_graph_true, NUM_INITIAL_OBSERVATIONS, num_iterations, INTERVENTION_COST, 'min', None, f_init)) for source in sources]
                        
                        results = [f.result() for f in futures]
                        for i in range(len(results)):
                                cost_results[i] = results[i][0]
                                opt_results[i] = results[i][1]

                        
                        tot_opts = []
                        tot_costs = []

                        if minimize:
                                cur_max = float('inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] <= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])

                                        tot_opts.append(min(opts))
                                        tot_costs.append(sum(costs))
                                        
                        else:
                                cur_max = float('-inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] >= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                                        
                                        tot_opts.append(max(opts))
                                        tot_costs.append(sum(costs))
                        
                        run_opts_cbo_single[n] = tot_opts
                        run_costs_cbo_single[n] = tot_costs 

                print(f"Run {n + 1} completed")
                n += 1

        except Exception as e:
                print(f"Run abandoned because of exception: {e}")
                fails +=1
                if fails > max_fails:
                        break
        





Causal mechanisms pre-fitted, skipping auto assignment


Fitting causal mechanism of node STATIN: 100%|██████████| 6/6 [00:00<00:00, 411.44it/s]


Causal mechanisms pre-fitted, skipping auto assignment


Fitting causal mechanism of node STATIN: 100%|██████████| 6/6 [00:00<00:00, 348.62it/s]


Causal mechanisms pre-fitted, skipping auto assignment


...

Iteration 1
Current global optimal set-value-result = ['None']: None -> 7.734206199645996
Performing parallel optimization source 0...
Performing parallel optimization source 1...


Iteration 2
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0914, 0.9422], dtype=torch.float64) -> 5.818776607513428
Performing parallel optimization source 0...
Performing parallel optimization source 1...
Iteration 3
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0980, 0.8878], dtype=torch.float64) -> 4.5427985191345215
Performing parallel optimization source 0...
Performing parallel optimization source 1...
Iteration 4
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0980, 0.8878], dtype=torch.float64) -> 4.5427985191345215
Performing parallel optimization source 0...
Performing parallel optimization source 1...
Iteration 5
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0980, 0.8878], dtype=torch.float64) -> 4.5427985191345215
Performing parallel optimization source 0...
Performing parallel optimization source 1...
Iteration 6
Curre

Fitting causal mechanism of node STATIN: 100%|██████████| 6/6 [00:00<00:00, 140.35it/s]

Iteration 1
Current global optimal set-value-result = ['None']: None -> 7.734206199645996
Intervening...
Iteration 1
Current global optimal set-value-result = ['None']: None -> 7.734206199645996
Intervening...


Iteration 2
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0804, 0.7941], dtype=torch.float64) -> 6.670061111450195
Intervening...
Iteration 2
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0032, 0.4007], dtype=torch.float64) -> 5.328632831573486
Intervening...
Iteration 3
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0240, 0.0150], dtype=torch.float64) -> 5.691049575805664
Intervening...
Iteration 3
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0032, 0.4007], dtype=torch.float64) -> 5.328632831573486
Intervening...
Iteration 4
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0813, 0.9085], dtype=torch.float64) -> 5.393269062042236
Intervening...
Iteration 4
Current global optimal set-value-result = frozenset({'STATIN', 'ASPIRIN'}): tensor([0.0575, 0.8039], dtype=torch.float64) -> 5.19463014602

In [ ]:
try:
    generate_trend_plot(run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single, graph=toy_graph, opt= opt, maximize=False)
    savepoint = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]
except Exception as e:
    print(e)
    savepoint = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]




## Scenario 1: Same DAG, different coefficients and noise

In [ ]:
from causalbo.cbo_loop_multisource import CBOLoopMultiSource
from causalbo.mcbo import multi_source_optimization
from matplotlib import pyplot as plt
from causalbo.sample_data.bi_graph import BiGraph
from causalbo.sample_data.psa_graph import PSAGraph
from copy import deepcopy
import time
import warnings
warnings.filterwarnings('ignore')

NUM_INITIAL_OBSERVATIONS = 30 
INTERVENTION_COST = 20
OBSERVATION_COST = 1

# PSAGraph object is a prebuilt class containing the medical dataset.
toy_graph = PSAGraph()
toy_graph_2 = PSAGraph()

# Copy observations to the other graph
# toy_graph_2.observational_samples = toy_graph.observational_samples
num_iterations = 10 

n_runs = 10
n = 0
run_ss_cbo = True # Possibility to run single-source CBO as well as a means of comparison

run_opts_cbo = [None] * n_runs
run_costs_cbo = [None] * n_runs
run_opts_std = [None] * n_runs
run_costs_std = [None] * n_runs
run_opts_cbo_single = [None] * n_runs
run_costs_cbo_single = [None] * n_runs

sources = [toy_graph, toy_graph_2]  # Assume 2 different information sources
source_fidelities = [1,1]
num_iterations = 10  # Number of iterations or total evaluations
max_fails = 5 # Maximum number of failed runs
fails = 0

# We wrap the method in a try except block to resolve stability issues and make sure we get enough runs in
while n < n_runs:
        try:
                # Store total cost
                total_cost_standard = NUM_INITIAL_OBSERVATIONS
                # Store changes in cost over time
                cost_over_time_standard = []
                # Store optimum over time
                global_optimum_over_time_standard = []

                t0 = time.time()
                # Initialize and run Causal GP using the CBO algorithm from Aglietti et. al.
                (global_optimum, global_optimal_set, global_optimal_source, gp, D_i, D_o, cost_over_time_causal, global_optimum_over_time_causal) = CBOLoopMultiSource(
                        observational_samples=[toy_graph.observational_samples, toy_graph_2.observational_samples],
                        graphs=[toy_graph.graph,toy_graph_2.graph],
                        source_costs = source_fidelities,
                        exploration_sets=[[frozenset(toy_graph.interventional_domain.keys())],[frozenset(toy_graph_2.interventional_domain.keys())]],
                        num_steps=num_iterations,
                        num_initial_obs=[NUM_INITIAL_OBSERVATIONS, NUM_INITIAL_OBSERVATIONS],
                        observation_cost=OBSERVATION_COST,
                        intervention_cost=INTERVENTION_COST,
                        num_obs_per_step=20,
                        num_max_allowed_obs=1000,
                        interventional_domains=[toy_graph.interventional_domain,toy_graph_2.interventional_domain],
                        type_trial='min',
                        pomis_criterion='distance',
                        objective_function=toy_graph_true,
                        early_stopping_iters=10, verbose=False, preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200)

                idx = global_optimum_over_time_causal.index(global_optimum) + 1
                global_optimum_over_time_causal = [global_optimum_over_time_causal[0]] + global_optimum_over_time_causal[:idx]
                cost_over_time_causal = [0] + cost_over_time_causal[:idx]

                run_opts_cbo[n] = global_optimum_over_time_causal
                run_costs_cbo[n] = cost_over_time_causal

                t1 = time.time()
                print(f'\nTotal time elapsed in seconds: {t1-t0}')
                print(f'\nOptimal source after BO = {global_optimal_source}')

                # Multi-source misokg
                (multi_source_gp, global_optimum_over_time_standard, cost_over_time_standard) = multi_source_optimization(
                        sources=sources, 
                        source_costs=source_fidelities, 
                        ground_truth=toy_graph_true,
                        budget=num_iterations, 
                        num_obs=NUM_INITIAL_OBSERVATIONS,
                        intervention_cost=INTERVENTION_COST,
                        total_cost_standard=[NUM_INITIAL_OBSERVATIONS,NUM_INITIAL_OBSERVATIONS],
                        type_trial='min',
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200
                        )


                run_opts_std[n] = global_optimum_over_time_standard
                run_costs_std[n] = cost_over_time_standard

                # Single source CBO
                if run_ss_cbo:
                        cost_results = [None] * len(sources)
                        opt_results = [None] * len(sources)
                        with concurrent.futures.ThreadPoolExecutor() as executor:
                                futures = [executor.submit(optim_threaded, (source, toy_graph_true, NUM_INITIAL_OBSERVATIONS, num_iterations, INTERVENTION_COST, 'min', [source.interventional_domain.keys()], f_init)) for source in sources]
                        
                        results = [f.result() for f in futures]
                        for i in range(len(results)):
                                cost_results[i] = results[i][0]
                                opt_results[i] = results[i][1]

                        
                        tot_opts = []
                        tot_costs = []

                        if minimize:
                                cur_max = float('inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] <= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                
                                        tot_opts.append(min(opts))
                                        tot_costs.append(sum(costs))
                                        
                        else:
                                cur_max = float('-inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] >= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                                        
                                        tot_opts.append(max(opts))
                                        tot_costs.append(sum(costs))
                        
                        run_opts_cbo_single[n] = tot_opts
                        run_costs_cbo_single[n] = tot_costs 

                print(f"Run {n + 1} completed")
                n += 1

        except Exception as e:
                print(f"Run abandoned because of exception: {e}")
                fails +=1
                if fails > max_fails:
                        break


In [ ]:
try:
    generate_trend_plot(run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single, graph=toy_graph, opt= opt, maximize=False)
    savepoint_2 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]
except Exception as e:
    savepoint_2 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]

## Scenario 2: Same DAGS's, different Structural Equation Models (different connections)

In [ ]:
from causalbo.cbo_loop_multisource import CBOLoopMultiSource
from causalbo.mcbo import multi_source_optimization
from matplotlib import pyplot as plt
from causalbo.sample_data.bi_graph import BiGraph
from causalbo.sample_data.psa_graph_s2 import PSAGraph
from causalbo.do_calculus import SCM
from copy import deepcopy
import time
import torch
import warnings
from pandas import DataFrame
warnings.filterwarnings('ignore')

NUM_INITIAL_OBSERVATIONS = 30 
INTERVENTION_COST = 20
OBSERVATION_COST = 1

# PSAGraph object is a prebuilt class containing the medical dataset.
toy_graph = PSAGraph()
toy_graph_2 = PSAGraph()

num_iterations = 10 

n_runs = 10
n = 0
run_ss_cbo = True # Possibility to run single-source CBO as well as a means of comparison

run_opts_cbo = [None] * n_runs
run_costs_cbo = [None] * n_runs
run_opts_std = [None] * n_runs
run_costs_std = [None] * n_runs
run_opts_cbo_single = [None] * n_runs
run_costs_cbo_single = [None] * n_runs

sources = [toy_graph, toy_graph_2]  # Assume 2 different information sources
source_fidelities = [1,1]
num_iterations = 10  # Number of iterations or total evaluations
max_fails = 5 # Maximum number of failed runs
fails = 0

# We wrap the method in a try except block to resolve stability issues and make sure we get enough runs in
while n < n_runs:
        try:
                # Store total cost
                total_cost_standard = NUM_INITIAL_OBSERVATIONS
                # Store changes in cost over time
                cost_over_time_standard = []
                # Store optimum over time
                global_optimum_over_time_standard = []

                t0 = time.time()
                # Initialize and run Causal GP using the CBO algorithm from Aglietti et. al.
                (global_optimum, global_optimal_set, global_optimal_source, gp, D_i, D_o, cost_over_time_causal, global_optimum_over_time_causal) = CBOLoopMultiSource(
                        observational_samples=[toy_graph.observational_samples, toy_graph_2.observational_samples],
                        graphs=[toy_graph.graph,toy_graph_2.graph],
                        source_costs = source_fidelities,
                        exploration_sets=[[frozenset(toy_graph.interventional_domain.keys())],[frozenset(toy_graph_2.interventional_domain.keys())]],
                        num_steps=num_iterations,
                        num_initial_obs=[NUM_INITIAL_OBSERVATIONS, NUM_INITIAL_OBSERVATIONS],
                        observation_cost=OBSERVATION_COST,
                        intervention_cost=INTERVENTION_COST,
                        num_obs_per_step=20,
                        num_max_allowed_obs=1000,
                        interventional_domains=[toy_graph.interventional_domain,toy_graph_2.interventional_domain],
                        type_trial='min',
                        pomis_criterion='distance',
                        objective_function=toy_graph_true,
                        early_stopping_iters=10, verbose=False,
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200)

                idx = global_optimum_over_time_causal.index(global_optimum) + 1
                global_optimum_over_time_causal = [global_optimum_over_time_causal[0]] + global_optimum_over_time_causal[:idx]
                cost_over_time_causal = [0] + cost_over_time_causal[:idx]

                run_opts_cbo[n] = global_optimum_over_time_causal
                run_costs_cbo[n] = cost_over_time_causal

                t1 = time.time()
                print(f'\nTotal time elapsed in seconds: {t1-t0}')
                print(f'\nOptimal source after BO = {global_optimal_source}')

                # Multi-source misokg
                (multi_source_gp, global_optimum_over_time_standard, cost_over_time_standard) = multi_source_optimization(
                        sources=sources, 
                        source_costs=source_fidelities, 
                        ground_truth=toy_graph_true,
                        budget=num_iterations, 
                        num_obs=NUM_INITIAL_OBSERVATIONS,
                        intervention_cost=INTERVENTION_COST,
                        total_cost_standard=[NUM_INITIAL_OBSERVATIONS,NUM_INITIAL_OBSERVATIONS],
                        type_trial='min',
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200
                        )


                run_opts_std[n] = global_optimum_over_time_standard
                run_costs_std[n] = cost_over_time_standard

                # Single source CBO
                if run_ss_cbo:
                        cost_results = [None] * len(sources)
                        opt_results = [None] * len(sources)
                        with concurrent.futures.ThreadPoolExecutor() as executor:
                                futures = [executor.submit(optim_threaded, (source, toy_graph_true, NUM_INITIAL_OBSERVATIONS, num_iterations, INTERVENTION_COST, 'min', [source.interventional_domain.keys()], f_init)) for source in sources]
                        
                        results = [f.result() for f in futures]
                        for i in range(len(results)):
                                cost_results[i] = results[i][0]
                                opt_results[i] = results[i][1]

                        
                        tot_opts = []
                        tot_costs = []

                        if minimize:
                                cur_max = float('inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] <= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                
                                        tot_opts.append(min(opts))
                                        tot_costs.append(sum(costs))
                                        
                        else:
                                cur_max = float('-inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] >= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                                        
                                        tot_opts.append(max(opts))
                                        tot_costs.append(sum(costs))
                        
                        run_opts_cbo_single[n] = tot_opts
                        run_costs_cbo_single[n] = tot_costs 

                print(f"Run {n + 1} completed")
                n += 1

        except Exception as e:
                print(f"Run abandoned because of exception: {e}")
                fails +=1
                if fails > max_fails:
                        break




In [ ]:
try:
    generate_trend_plot(run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single, graph=toy_graph, opt= opt, maximize=False)
    savepoint_3 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]
except Exception as e:
    savepoint_3 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]

## Scenario 3: Different DAGS's, different Structural Equation Models (nodes added / deleted)

In [ ]:
from causalbo.cbo_loop_multisource import CBOLoopMultiSource
from causalbo.mcbo import multi_source_optimization
from matplotlib import pyplot as plt
from causalbo.sample_data.bi_graph import BiGraph
from causalbo.sample_data.psa_graph_s3 import PSAGraph
from causalbo.do_calculus import SCM
from copy import deepcopy
import time
import torch
import warnings
from pandas import DataFrame
warnings.filterwarnings('ignore')
NUM_INITIAL_OBSERVATIONS = 30 
INTERVENTION_COST = 20
OBSERVATION_COST = 1

# PSAGraph object is a prebuilt class containing the medical dataset.
toy_graph = PSAGraph()
toy_graph_2 = PSAGraph()

num_iterations = 10 

n_runs = 10
n = 0
run_ss_cbo = True # Possibility to run single-source CBO as well as a means of comparison

run_opts_cbo = [None] * n_runs
run_costs_cbo = [None] * n_runs
run_opts_std = [None] * n_runs
run_costs_std = [None] * n_runs
run_opts_cbo_single = [None] * n_runs
run_costs_cbo_single = [None] * n_runs

sources = [toy_graph, toy_graph_2]  # Assume 2 different information sources
source_costs = [INTERVENTION_COST,INTERVENTION_COST]
num_iterations = 10  # Number of iterations or total evaluations
max_fails = 5 # Maximum number of failed runs
source_fidelities = [1,1]
fails = 0

# We wrap the method in a try except block to resolve stability issues and make sure we get enough runs in
while n < n_runs:
        try:
                # Store total cost
                total_cost_standard = NUM_INITIAL_OBSERVATIONS
                # Store changes in cost over time
                cost_over_time_standard = []
                # Store optimum over time
                global_optimum_over_time_standard = []

                t0 = time.time()
                # Initialize and run Causal GP using the CBO algorithm from Aglietti et. al.
                (global_optimum, global_optimal_set, global_optimal_source, gp, D_i, D_o, cost_over_time_causal, global_optimum_over_time_causal) = CBOLoopMultiSource(
                        observational_samples=[toy_graph.observational_samples, toy_graph_2.observational_samples],
                        graphs=[toy_graph.graph,toy_graph_2.graph],
                        source_costs = source_fidelities,
                        exploration_sets=[[frozenset(toy_graph.interventional_domain.keys())],[frozenset(toy_graph_2.interventional_domain.keys())]],
                        num_steps=num_iterations,
                        num_initial_obs=[NUM_INITIAL_OBSERVATIONS, NUM_INITIAL_OBSERVATIONS],
                        observation_cost=OBSERVATION_COST,
                        intervention_cost=INTERVENTION_COST,
                        num_obs_per_step=20,
                        num_max_allowed_obs=1000,
                        interventional_domains=[toy_graph.interventional_domain,toy_graph_2.interventional_domain],
                        type_trial='min',
                        pomis_criterion='distance',
                        objective_function=toy_graph_true,
                        early_stopping_iters=10, verbose=False, 
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200)

                idx = global_optimum_over_time_causal.index(global_optimum) + 1
                global_optimum_over_time_causal = [global_optimum_over_time_causal[0]] + global_optimum_over_time_causal[:idx]
                cost_over_time_causal = [0] + cost_over_time_causal[:idx]

                run_opts_cbo[n] = global_optimum_over_time_causal
                run_costs_cbo[n] = cost_over_time_causal

                t1 = time.time()
                print(f'\nTotal time elapsed in seconds: {t1-t0}')
                print(f'\nOptimal source after BO = {global_optimal_source}')

                # Multi-source misokg
                (multi_source_gp, global_optimum_over_time_standard, cost_over_time_standard) = multi_source_optimization(
                        sources=sources, 
                        source_costs=source_fidelities, 
                        ground_truth=toy_graph_true,
                        budget=num_iterations, 
                        num_obs=NUM_INITIAL_OBSERVATIONS,
                        intervention_cost=INTERVENTION_COST,
                        total_cost_standard=[NUM_INITIAL_OBSERVATIONS,NUM_INITIAL_OBSERVATIONS],
                        type_trial='min',
                        preset_opt=f_init,
                        cutoff_criterion="cost", cost_cutoff=1200
                        )


                run_opts_std[n] = global_optimum_over_time_standard
                run_costs_std[n] = cost_over_time_standard

                # Single source CBO
                if run_ss_cbo:
                        cost_results = [None] * len(sources)
                        opt_results = [None] * len(sources)
                        with concurrent.futures.ThreadPoolExecutor() as executor:
                                futures = [executor.submit(optim_threaded, (source, toy_graph_true, NUM_INITIAL_OBSERVATIONS, num_iterations, INTERVENTION_COST, 'min', [source.interventional_domain.keys()], f_init)) for source in sources]
                        
                        results = [f.result() for f in futures]
                        for i in range(len(results)):
                                cost_results[i] = results[i][0]
                                opt_results[i] = results[i][1]

                        
                        tot_opts = []
                        tot_costs = []

                        if minimize:
                                cur_max = float('inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] <= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                
                                        tot_opts.append(min(opts))
                                        tot_costs.append(sum(costs))
                                        
                        else:
                                cur_max = float('-inf')
                                for i in range(0, max(len(opt_result) for opt_result in opt_results)):
                                        opts = []
                                        costs = []
                                        for j in range(len(results)):
                                                if i < len(opt_results[j]):
                                                        costs.append(cost_results[j][i])
                                                        if opt_results[j][i] >= cur_max:
                                                                opts.append(opt_results[j][i])
                                                                cur_max = opt_results[j][i]
                                                        else:
                                                                opts.append(cur_max)
                                                else:
                                                        costs.append(cost_results[j][len(cost_results[j]) - 1])
                                        
                                        tot_opts.append(max(opts))
                                        tot_costs.append(sum(costs))
                        
                        run_opts_cbo_single[n] = tot_opts
                        run_costs_cbo_single[n] = tot_costs 

                print(f"Run {n + 1} completed")
                n += 1

        except Exception as e:
                print(f"Run abandoned because of exception: {e}")
                fails +=1
                if fails > max_fails:
                        break




In [ ]:
try:
    generate_trend_plot(run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single, graph=toy_graph, opt= opt, maximize=False)
    savepoint_4 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]
except Exception as e:
    savepoint_4 = [run_costs_cbo,run_opts_cbo,run_costs_std,run_opts_std, run_costs_cbo_single, run_opts_cbo_single]